# Time-dependent PDEs in FEniCSx

## Time is crucial for many processes

In [ ]:
from IPython.display import YouTubeVideo

YouTubeVideo("CCmTY0PKGDs", width=560, height=316)

## The heat equation is the "Hello world" example of a time-dependent PDE

We will solve the simplest extension of the Poisson problem into
the time domain, the **heat equation**:

\begin{align*}
\frac{\partial u}{\partial t} + D u_{xx} &= f \qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T], \\
u &= g \qquad \text{ for } x \in \partial \Omega \text{ and } t \in [0, T], \\
u &= u^0 \qquad \text{ for } x \in \Omega \text{ and } t = 0.
\end{align*}

Here, $D \in \mathbb R$. $\Omega$ denotes the spatial domain and $\partial \Omega$ the domain boundary. For example, for $\Omega = [-L, L]$, the boundary $\partial \Omega$ consists of the two points $-L$ and $L$. We start the simulation at time $t=0$ and run it until the final time $T$.


The solution $u = u(x, t)$, the right-hand side $f = f (x, t)$, and the
boundary value $g = g(x, t)$ may vary in space $(x)$
and time $(t)$. The initial value $u_0$ is a function of space only.

Let's see what a 2D solution of this problem looks like:

In [ ]:
from IPython.display import YouTubeVideo

YouTubeVideo("TvlIfSlLB0c", width=560, height=316)

**Question**: Which kind of initial/boundary conditions were used in the above simulation?

## Time-discretization of the heat equation

FEniCS cannot direct solve time-dependent equations. We therefore must approximate the time-derivative $\frac{\partial u}{\partial t}$; that is, we discretize in time. This will lead to a sequence of time-independent PDEs, which we *can* solve with FEniCS. 


Our goal is to compute the solution at a set of discrete time-levels $0 = t^0 < t^1 < ... < T^N = T$. 
The variables at the $n$-th time-level will be denoted with a superscript $n$:

$$ u^n \approx u(t^n) \\
f^n = f(t^n)
$$

We discretize in time using the **implicit Euler** method.

$$ \frac{\partial u}{\partial t} (t^n) \approx \frac{u^n - u^{n-1}}{\Delta t} $$
This leads to the semi-discretization of the heat equation ("semi"-discretization because we have discretized in time, but not yet in space):
$$
u^n - u^{n-1} - \Delta t D u^n_{xx} = \Delta t f_n 
$$


### Time stepping algorithm for the heat equation

1. Start with $u^0$ and choose a time step $\Delta t$ > 0.
2. For $n = 1, 2, ...$, solve for $u^n$: 
 
   $u^n − \Delta t D u^n_{xx} = u^{n-1} + \Delta t f^n$

## Variational problem for the heat equation

To obtain the fully-discretized system, we derive the variational formulation of the semi-discretized system:

Find $u^n, n=1, ..., N$ such that 
$$
 a(u^n, v) = L^n(v)  \quad \text{ for all } v
$$
where 
$$
  a(u^n, v) = \int_\Omega u^nv + D \Delta t u_x v_x \text{d}x\\
  L^n(v) = \int_\Omega u^{n-1}v + \Delta t f^nv \text{d}x
$$
Note that the bilinear form $a(u, v)$ is constant while the linear
form $L^n$ depends on $n$.

## Detailed time-stepping algorithm for the heat equation

The following skeleton shows how to solve a time-dependent problem in FEniCS:

* Define the mesh, functions spaces and Dirichlet boundary conditions
* Compute $u_0$ as the projection of the given initial value
* Define the forms $a$ and $L$
* Set $t=\Delta t$
* **while $t \leq T$ do**
    * Apply the boundary condition
    * Solve the $AU = L$ for $U$ and store in $u_1$
    * Set $t$ to $t + \Delta t$
    * Set $u_0 = u_1$ (get ready for next step)
* **end while**

## Some implementation tips

In [ ]:
from dolfinx import fem, mesh
from dolfinx.fem.petsc import (
    assemble_matrix,
    assemble_vector,
    apply_lifting,
    set_bc,
    create_vector,
)
from petsc4py import PETSc
from ufl import TestFunction, TrialFunction, dx, grad, inner
from mpi4py import MPI
import numpy as np

domain = mesh.create_unit_interval(MPI.COMM_WORLD, 10)
V = fem.functionspace(domain, ("CG", 1))

## Handling time-dependent expressions

We need to define a time-dependent expression for the boundary value:

In [ ]:
class BoundaryValue:
    def __init__(self, beta: float, t: float):
        self.beta = beta
        self.t = t

    def __call__(self, x):
        return 1 - self.beta * self.t + 0 * x[0]


beta = 1.2
t = 0.0
g = BoundaryValue(beta=beta, t=t)

## Implementing the variational problem

In [ ]:
# Time step
dt = 0.1

# Initial condition
u_D = fem.Function(V)
u_D.interpolate(g)

# Define a variable to store the solution at the previous time-step
u_n = fem.Function(V)
u_n.x.array[:] = u_D.x.array[:]

# Define a source term
f = fem.Constant(domain, 0.0)


# Define the Dirichlet boundary condition
def on_boundary(x):
    return np.isclose(x[0], 0) | np.isclose(x[0], 1)


boundary_facets = mesh.locate_entities_boundary(
    domain, domain.topology.dim - 1, on_boundary
)
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)

bc = fem.dirichletbc(u_D, boundary_dofs)

# Code the variational formulation
u = TrialFunction(V)
v = TestFunction(V)

a = u * v * dx + dt * inner(grad(u), grad(v)) * dx
L = u_n * v * dx + dt * f * v * dx

a = fem.form(a)
L = fem.form(L)

In [ ]:
A = assemble_matrix(a, bcs=[bc])
A.assemble()
b = create_vector(fem.extract_function_spaces(L))
uh = fem.Function(V)

In [ ]:
solver = PETSc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(PETSc.KSP.Type.PREONLY)
pc = solver.getPC()
pc.setType(PETSc.PC.Type.LU)

## Implementing the time-stepping loop

In [ ]:
from matplotlib import pyplot as plt

t = 0
T = 1.0
g.t = t

x_coords = V.tabulate_dof_coordinates()[:, 0]
plt.figure(figsize=(8, 6))
plt.plot(x_coords, u_n.x.array, label=f"t={t:.1f} (IC)")


while t < T:
    # Update Diriclet boundary condition
    t += dt
    g.t += dt
    u_D.interpolate(g)

    # Update the right hand side reusing the initial vector
    with b.localForm() as loc_b:
        loc_b.set(0)
    assemble_vector(b, L)

    # Apply Dirichlet boundary condition to the vector
    apply_lifting(b, [a], [[bc]])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, [bc])

    # Solve linear problem
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # Update solution at previous time step (u_n)
    u_n.x.array[:] = uh.x.array

    # Plot current step
    plt.plot(x_coords, uh.x.array, label=f"t={t:.2f} (g={1 - g.beta*g.t:.2f})")

# Finalize Plot
plt.xlabel("x")
plt.ylabel("u")
plt.legend()
plt.title("Heat Equation")
plt.grid(True)
plt.show()

## The cable equations

The standard cable equation is a reaction-diffusion equation given by
$$
\frac{\partial u}{\partial t}  = \sigma u_{xx} + f(u, s)
$$
where $f(u, s)$ is a reaction term describing ionic fluxes across
the membrane.
* A linear $f(u)$ describes passive conductance through a leaky cable (dendrites).
* A cubic $f(u)$ gives the bistable equation with a propagating activation front.
* In general $f(u, s)$, where s is a vector describing the state of the cell membrane, typically governed by a system of ODEs.

## Exercise 1: The cable equation

Solve the linear, bistable cable equation on an interval $\Omega=[-L, L]$ in FEniCS 
\begin{align*}
\frac{\partial u}{\partial t} &= \sigma u_{xx} + f(u) \quad &&\text{ for } -L < x < L, \\
u_x &= 0 \quad &&\text{ for } x = -L \text{ and } x = L,
\end{align*}
with 
* $f(u) = Au$,
* $A = -0.1$,
* $\sigma = 1.0$,
* $L = 100$.

Implement an implicit Euler time-stepping scheme and solve the problem from $t=0$ to $T=250$ with a time step of $dt=2.5$. Use as initial condition

$$
u(x,0) = \frac{1}{2} \left( 1 - \tanh\left(\sqrt{\frac{-A}{8\sigma}} (x + 0.75L)\right) \right)
$$
Create a plot of the solution on every 10th time-step. What happens if you change the value and sign of A? To improve the performance of your solver, try making it assemble `A` only once.

## Exercise 1 solution

Code doing this is below.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc
from ufl import (
    SpatialCoordinate,
    tanh,
    sqrt,
    TestFunction,
    TrialFunction,
    dx,
    inner,
    grad,
)
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, create_vector
import matplotlib.pyplot as plt

# Setup Mesh and Function Space
# IntervalMesh(20, -100, 100) becomes:
domain = mesh.create_interval(MPI.COMM_WORLD, 20, [-100.0, 100.0])
V = fem.functionspace(domain, ("CG", 1))

# Parameters
t = 0.0
T = 250.0
dt = 2.5
L_val = 100.0
sigma = fem.Constant(domain, 1.0)
A = fem.Constant(domain, -0.1)

# Initial condition
u0 = fem.Function(V)
x = SpatialCoordinate(domain)
ufl_expr = 0.5 * (1.0 - tanh(sqrt(-A / (8.0 * sigma)) * (x[0] + 0.75 * L_val)))
expr = fem.Expression(ufl_expr, V.element.interpolation_points)
u0.interpolate(expr)

# Variational formulation
u1 = fem.Function(V)  # The unknown at the new time step
u = TrialFunction(V)
v = TestFunction(V)

a = u * v * dx + dt * sigma * inner(grad(u), grad(v)) * dx - dt * A * u * v * dx
L = u0 * v * dx

bilinear_form = fem.form(a)
linear_form = fem.form(L)

# Pre-assemble the matrix and configure the Solver
# Since 'a' is time-independent, we assemble A once to save computation time.
A_mat = assemble_matrix(bilinear_form)
A_mat.assemble()

solver = PETSc.KSP().create(domain.comm)
solver.setOperators(A_mat)
solver.setType(PETSc.KSP.Type.PREONLY)
solver.getPC().setType(PETSc.PC.Type.LU)

b = create_vector(fem.extract_function_spaces(linear_form))

# Prepare coordinates for plotting
x_coords = V.tabulate_dof_coordinates()[:, 0]

plt.figure(figsize=(10, 6))

n_timesteps = 0

while t < T:
    n_timesteps += 1
    t += dt

    # Reset RHS vector and re-assemble
    with b.localForm() as loc_b:
        loc_b.set(0)
    assemble_vector(b, linear_form)

    # Note: No Dirichlet BCs in this formulation, so we skip apply_lifting and set_bc
    b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)

    # Solve the linear system
    solver.solve(b, u1.x.petsc_vec)
    u1.x.scatter_forward()

    # Update u0 for the next time-step
    u0.x.array[:] = u1.x.array

    # Plot every 10 steps
    if n_timesteps % 10 == 0:
        plt.plot(x_coords, u0.x.array, label=f"t={t:.1f}")

# Finalize Plot
plt.xlabel("x")
plt.ylabel("u")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.title("Cable Equation")
plt.grid(True)
plt.tight_layout()
plt.show()

## Exercise 2: Forward or backward?

Above, we discretized in time using the **implicit Euler** method, meaning that we replaced $ \frac{\partial u}{\partial t} (t^n)$ by $\frac{u^n - u^{n-1}}{\Delta t} $. This is also called the **backward** Euler method because we are approximating a derivative by a difference *backwards* in time.

We could also have used the **forward** or **explicit Euler** method by approximating $ \frac{\partial u}{\partial t} (t^n)$ by $\frac{u^{n+1} - u^n}{\Delta t} $. Note that now the difference is going *forwards* in time instead. Take the heat equation $$\frac {\partial u} {\partial t} = Du_{xx}$$ and discretize it using both methods. What does the resulting systems of equations look like? Why do you think one is called explicit and the other implicit?

Next, let us try to compare the two. Choose initial conditions and Dirichlet boundary conditions so that the exact solution becomes $u_e = \text{exp}(-t) \text{ sin}(x)$. Then, using a time step of $\Delta t$ = 0.1, derive the weak formulation and solve using first a backward Euler discretization, then a forward Euler discretization. Plot the computed solution alongside the exact solution, or compute the error. Which time discretization would you prefer?

## Exercise 2 solution

With implicit Euler, the weak form becomes

$\int u^n v + D \Delta t \: u^n_x v_x = \int u^{n-1} v$ 

With explicit Euler, it is

$\int u^n v = \int u^{n-1} v - D \Delta t \: u^{n-1}_x v_x $ 

So in a way, in the latter case the resulting system of equations is basically $u^n = $ something known, so you don't really need to solve an equation for $u^n$ at all - it is **explicitly** given. The below code solves using explicit Euler (Uncomment the two lines starting with `a`, `L` for implicit Euler). This results in numerical artifacts and huge oscillations when the time step is too big. Backward Euler may also be inaccurate, but is guaranteed to be stable (for this problem, at least), so is in general preferable. Of course, that comes at the cost of being harder to compute, so in applications where performance is the most important, it might conceivably be better to use explicit Euler.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc
from ufl import (
    SpatialCoordinate,
    exp,
    sin,
    TrialFunction,
    TestFunction,
    dx,
    inner,
    grad,
)
from dolfinx import fem, mesh
import matplotlib.pyplot as plt

# Setup mesh and function space
domain = mesh.create_unit_interval(MPI.COMM_WORLD, 20)
V = fem.functionspace(domain, ("CG", 1))

# Parameters
dt_val = 0.1
T = 1.0
t = 0.0
D = fem.Constant(domain, 1.0)
dt = fem.Constant(domain, dt_val)
t_const = fem.Constant(domain, t)

# Exact solution and initial condition
x = SpatialCoordinate(domain)
# Define exact solution symbolically using UFL
u_exact_ufl = exp(-t_const) * sin(x[0])

# Create a FEniCSx Function to store the exact solution (used for BCs)
u_exact_func = fem.Function(V)
expr = fem.Expression(u_exact_ufl, V.element.interpolation_points)
u_exact_func.interpolate(expr)

u_prev = fem.Function(V)
u_prev.interpolate(expr)  # Initial condition at t=0

# Boundary conditions
domain.topology.create_connectivity(domain.topology.dim - 1, domain.topology.dim)
boundary_facets = mesh.exterior_facet_indices(domain.topology)
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)

bc = fem.dirichletbc(u_exact_func, boundary_dofs)

# Variational formulation
u = TrialFunction(V)
v = TestFunction(V)
uh = fem.Function(V)


## Explicit Euler
a = u * v * dx
L = u_prev * v * dx - dt * D * inner(grad(u_prev), grad(v)) * dx

## Implicit Euler (Uncomment to use)
# a = u * v * dx + dt * D * inner(grad(u), grad(v)) * dx
# L = u_prev * v * dx

bilinear_form = fem.form(a)
linear_form = fem.form(L)

# Pre-assemble the matrix
A = fem.petsc.assemble_matrix(bilinear_form, bcs=[bc])
A.assemble()

solver = PETSc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(PETSc.KSP.Type.PREONLY)
solver.getPC().setType(PETSc.PC.Type.LU)

b = fem.petsc.create_vector(fem.extract_function_spaces(linear_form))

while t < T - 1e-8:
    t += dt_val

    # Update time-dependent exact solution (updates BC implicitly)
    t_const.value = t
    u_exact_func.interpolate(expr)

    # Assemble RHS vector
    with b.localForm() as loc_b:
        loc_b.set(0)
    fem.petsc.assemble_vector(b, linear_form)

    # Apply Dirichlet boundary conditions
    fem.petsc.apply_lifting(b, [bilinear_form], [[bc]])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    fem.petsc.set_bc(b, [bc])

    # Solve
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # Update previous step
    u_prev.x.array[:] = uh.x.array

# Plot the final solution vs exact
x_coords = V.tabulate_dof_coordinates()[:, 0]

plt.figure(figsize=(8, 5))
plt.plot(x_coords, uh.x.array, "ro-", label="Numerical")
plt.plot(x_coords, u_exact_func.x.array, "b--", label="Exact")
plt.xlabel("x")
plt.ylabel("u")
plt.title(f"Heat Equation at t={T}")
plt.legend()
plt.show()

## Exercise 3: 2D heat equation
(This exercise may be challenging without knowledge of vector calculus. Feel free to skip it.)

The heat equation can be extended to a 2D domain in the following way:
$$\frac{\partial u}{\partial t} - D \text{ div grad } u = f$$

Here, $\text{ grad }$ means the gradient (vector of $x$ and $y$ derivatives), and $\text{ div }$ means the divergence (sum of $x$ and $y$ derivatives).

To derive its weak formulation, you will need to generalize integration by parts to 2D. The relevant formula is:

$$ \int_\Omega \psi \text{ div grad } \phi = - \int_\Omega \text{grad } \psi \cdot \text{ grad } \phi  + \text{ boundary term} $$

Use this to derive the weak formulation of the 2D heat equation. Then solve the heat equation on a unit square with the Dirichlet boundary condition $u = x(1-x)$ on the entire boundary, and the initial condition $u=0$.

## Exercise 3 solution:



In [ ]:
from mpi4py import MPI
from petsc4py import PETSc
from ufl import TestFunction, TrialFunction, dx, grad, inner
from dolfinx import fem, mesh, plot
import pyvista

# Setup mesh and function space
domain = mesh.create_unit_square(MPI.COMM_WORLD, 10, 10)
V = fem.functionspace(domain, ("CG", 1))

# Parameters
dt = 0.1
T = 100.0
D = fem.Constant(domain, 1e-2)

# Define variables for previous and current solutions
u_prev = fem.Function(V)
uh = fem.Function(V)


# Boundary & initial conditions
def bdy_expression(x):
    return x[0] * (1.0 - x[0])


u_D = fem.Function(V)
u_D.interpolate(bdy_expression)


# Create connectivity to evaluate facets, then locate the exterior boundary dofs
domain.topology.create_connectivity(domain.topology.dim - 1, domain.topology.dim)
boundary_facets = mesh.exterior_facet_indices(domain.topology)
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)

bc = fem.dirichletbc(u_D, boundary_dofs)

# Variational formulation
u = TrialFunction(V)
v = TestFunction(V)

a = u * v * dx + dt * D * inner(grad(u), grad(v)) * dx
L = u_prev * v * dx

bilinear_form = fem.form(a)
linear_form = fem.form(L)

# Pre-assemble matrix and configure solver
# The LHS matrix 'A' does not change over time, so assemble it once outside the loop.
A = fem.petsc.assemble_matrix(bilinear_form, bcs=[bc])
A.assemble()

solver = PETSc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(PETSc.KSP.Type.PREONLY)
solver.getPC().setType(PETSc.PC.Type.LU)

# Create a vector for the RHS
b = fem.petsc.create_vector(fem.extract_function_spaces(linear_form))

t = 0.0

while t < T:
    t += dt

    # Reset RHS vector and assemble
    with b.localForm() as loc_b:
        loc_b.set(0)
    fem.petsc.assemble_vector(b, linear_form)

    # Apply lifting and boundary conditions to the RHS vector
    fem.petsc.apply_lifting(b, [bilinear_form], [[bc]])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    fem.petsc.set_bc(b, [bc])

    # Solve linear system
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # Assign current solution to previous for the next time step
    u_prev.x.array[:] = uh.x.array

grid = pyvista.UnstructuredGrid(*plot.vtk_mesh(domain))
grid.point_data["Temperature"] = uh.x.array
plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True)
plotter.view_xy()
plotter.show()